In [ ]:
from pprint import pprint

import matplotlib.pyplot as plt
import numpy as np

from mdx2.io import loadobj


# Scaling model

Visualize the mdx2 scaling model.

## Parameters

In [ ]:
# These metadata fields are injected by mdx2.report, regardless of the template. Don't change this line.
_metadata: dict = {}

# Notebook parameters
input_files: list[str] = None # list of input file paths: required
model_names: list[str] = ['scaling_model','offset_model','absorption_model','detector_model'] # models to include
shared_detector_model: bool = True # assume that all datasets have the same detector model (only display the first one).

In [ ]:
# pretty-print the metadata and parameters
pprint({
    "metadata": _metadata,
    "parameters": {"input_files": input_files, "model_names": model_names}
})

## Results

In [ ]:
Models = {fn: {v: loadobj(fn, v) for v in model_names} for fn in input_files}

### Scale

In [ ]:
if 'scaling_model' not in model_names:
    print("skipped")
else:
    for name, M in Models.items():
        b = M['scaling_model'].to_xarray()
        b.plot(aspect=1.5, size=3)
        plt.title(f"{name}: scaling_model")

### Offset

In [ ]:
if 'offset_model' not in model_names:
    print("skipped")
else:
    for name, M in Models.items():
        c = M['offset_model'].to_xarray()
        c.plot(aspect=1.5, size=3)
        plt.title(f"{name}: offset_model")

### Absorption

In [ ]:
if 'absorption_model' not in model_names:
    print("skipped")
else:
    for name, M in Models.items():
        a = M['absorption_model'].to_xarray()
        phi_min = a.phi[0].data
        phi_max = a.phi[-1].data
        phi_interval = 360/8
        npts = int(1 + (phi_max - phi_min) / phi_interval)
        col_wrap = min(8, npts)
        a_sliced = a.sel(phi=np.linspace(phi_min, phi_max, npts),method='nearest')
        g = a_sliced.plot(
            yincrease=False,
            x='ix',
            y='iy',
            col='phi',
            cmap='coolwarm',
            robust=True,
            col_wrap=col_wrap,
            size=1.5,
            )
        for ax in g.axs.flatten():
            ax.set_aspect('equal')
        print(f"{name}: absorption_model")
        plt.show()

### Detector

In [ ]:
if 'detector_model' not in model_names:
    print("skipped")
else:
    for name, M in Models.items():
        d = M['detector_model'].to_xarray()
        d.plot(x='ix', y='iy', yincrease=False,cmap='coolwarm', robust=True, size=6)
        if shared_detector_model:
            plt.title("detector_model (shared)")
            break
        else:
            plt.title(f"{name}: detector_model")